# Entrainer un modèle de classification

In [3]:
!pip install scikit-learn
!pip install ultralytics
!pip install tqdm
!pip install pandas
!pip install os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [4]:
import requests
import random
import os
import time
import shutil

from sklearn.model_selection import train_test_split
from tqdm import tqdm

## 1. Récupérer des annotations

Un classificateur répartit les documents d'un ensemble de données en catégories. Parmi les types de classificateurs possibles, on trouve le classificateur multi-étiquetage supervisé.

Ce classificateur répartit les documents dans des classes prédéfinies :
- Avec une illustration
- Sans illustration

Il utilise l'apprentissage automatique pour déterminer la classification des documents.

Ici les annotations sont en fait les dossiers dans lesquelles ont stock les images. Les classificateurs sont des modèles qui appliquent une étiquette à l'échelle de l'image entière et il est donc très simple de produire des données.

In [5]:
!apt-get install -y git-lfs > /dev/null
!git lfs install

!rm -rf formationVision
!git clone https://huggingface.co/datasets/Anarchiviste/formationVision

%cd formationVision
!git lfs pull
%cd ..

Git LFS initialized.
Cloning into 'formationVision'...
remote: Enumerating objects: 268, done.
remote: Total 268 (delta 0), reused 0 (delta 0), pack-reused 268 (from 1)
Receiving objects: 100% (268/268), 41.34 KiB | 3.76 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Filtering content: 100% (238/238), 67.32 MiB | 10.22 MiB/s, done.
/content/formationVision
/content


In [6]:
source = "./formationVision"

illustrations = os.listdir(os.path.join(source, 'illustrated/'))
textOnly = os.listdir(os.path.join(source, 'textonly/'))

trainIllustrations, valIllustrations = train_test_split(illustrations, train_size=0.8, random_state=42)
trainTextOnly, valTextOnly = train_test_split(textOnly, train_size=0.8, random_state=42)


In [7]:
!rm -rf dataset/
!mkdir -p ./dataset/train/illustrated/
!mkdir -p ./dataset/train/textonly/
!mkdir -p ./dataset/val/illustrated/
!mkdir -p ./dataset/val/textonly/


In [8]:
print(f"Nous aurons {len(trainIllustrations)} illustrations dans Train et {len(valIllustrations)} illustrations dans Val")
print(f"Nous aurons {len(trainTextOnly)} textes dans Train et {len(valTextOnly)} textes dans Val")

Nous aurons 72 illustrations dans Train et 18 illustrations dans Val
Nous aurons 76 textes dans Train et 19 textes dans Val


In [9]:
copies = [
    (trainIllustrations, 'illustrated', './dataset/train/illustrated/'),
    (valIllustrations, 'illustrated', './dataset/val/illustrated/'),
    (trainTextOnly, 'textonly', './dataset/train/textonly/'),
    (valTextOnly, 'textonly', './dataset/val/textonly/')
]

# Mise en place d'un dataset de Train et de Val
for files, folder_name, dest_dir in copies:
    for file_name in files:
        shutil.copy(
            os.path.join(source, folder_name, file_name),
            os.path.join(dest_dir, file_name)
        )

In [11]:
from ultralytics import YOLO

# Nous importons un modèle Yolo 26 nano classificateur avec la bibliothèque d'Ultralytics
model = YOLO("yolo26n-cls.pt")

In [13]:
dataset = "./dataset/"
model.train(data = dataset, flipud = 0, fliplr = 0, epochs = 10, imgsz = 640)

Ultralytics 8.4.202 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset/, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0, flipud=0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_m

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x794e87701310>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.0023950270416035722, 'inference': 92.31623659460072, 'loss': 7.302702903650022e-05, 'postprocess': 0.00019159459633236699}
top1: 1.0
top5: 1.0

# Faire tourner une inférence

In [14]:
# On charge le modèle avec les meilleurs valeures avant un surapprentissage (souvent nommé best)
best = YOLO("./runs/classify/train/weights/best.pt")

In [20]:
import os
import pandas as pd

# Dossier contenant les images de test
test_dir = "./formationVision/test/press/"

results_list = []
image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Exécuter l'inférence sur chaque image
for file_name in image_files:
    file_path = os.path.join(test_dir, file_name)
    results = best(file_path, verbose=False)

    for r in results:
        probs = r.probs
        top_class_idx = probs.top1
        top_class_name = r.names[top_class_idx]
        top_class_conf = float(probs.top1conf)

        results_list.append({
            "filename": file_name,
            "prediction": top_class_name,
            "confidence": top_class_conf
        })

# Convertir en DataFrame pandas et sauvegarder en CSV
df = pd.DataFrame(results_list)
df.sort_values(by='filename', ascending=False)
df.to_csv("predictions.csv", index=False)

print(f"Inférence terminée ! Les résultats ont été enregistrés dans 'predictions.csv'.")
print(df.head())

Inférence terminée ! Les résultats ont été enregistrés dans 'predictions.csv'.
  filename   prediction  confidence
0    4.png  illustrated    0.910941
1   37.png     textonly    0.986176
2   16.png     textonly    0.999589
3   14.png     textonly    0.981780
4   34.png     textonly    0.979701
